#### Libs

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

In [ ]:
import pandas as pd

from src.preprocessing.split import make_group_split
from src.preprocessing.transforms import get_transforms
from src.data.dataset import make_loaders
from src.models.model import get_model
from src.training.trainer import MRIQualityTrainer

import torch
import torch.nn as nn

#### Configs

In [ ]:
DF_PATH = Path('../data/demographics/matched_psm_info.csv')
IMAGES_DIR = Path('../data/raw/')
RESULTS_DIR = Path('../results/')

MODEL_NAME = 'resnet18'  # or 'densenet121' or 'efficientnet-b0'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LEARNING_RATE = 1e-4

BATCH_SIZE = 2
NUM_WORKERS = 4
PIN_MEMORY = True
NUM_EPOCHS = 10
N_SPLITS = 5
RANDOM_STATE = 42

#### Reading data

In [ ]:
df = pd.read_csv(DF_PATH, sep=';')
df.sample(n=3, random_state=RANDOM_STATE)

#### Dataprep

Generating splits with k-fold

In [ ]:
split_data = make_group_split(
    df=df,
    images_dir=IMAGES_DIR,
    split_type='kfold',
    verbose=True
)

Get transforms

In [ ]:
transforms = get_transforms()

#### Training

In [ ]:
fold_histories = []
test_data = split_data['test']

print(test_data[0])

In [ ]:
for fold in split_data['folds']:
    print(f"--- Fold {fold['fold']} ---")

    train_loader, val_loader = make_loaders(
        train_data=fold['train'],
        val_data=fold['val'],
        transforms=transforms,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle_train=True
    )

    model = get_model(model_name=MODEL_NAME)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    checkpoint_dir = RESULTS_DIR / f"fold_{fold['fold']}"

    trainer = MRIQualityTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
        checkpoint_dir=checkpoint_dir,
        monitor_metric='auc'
    )

    history = trainer.fit(num_epochs=NUM_EPOCHS, verbose=True)
    fold_histories.append(history)

#### Test set evaluation

In [ ]:
test_loader = make_loaders(
    train_data=test_data,
    val_data=test_data,
    transforms=transforms,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    shuffle_train=False,
)[1]

In [ ]:
test_results = []

for fold_idx in range(N_SPLITS):
    model = get_model(model_name=MODEL_NAME)
    checkpoint_path = RESULTS_DIR / \
        f"fold_{fold_idx}" / f"{model._get_name()}_best_model.pt"

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    trainer = MRIQualityTrainer(
        model=model,
        train_loader=test_loader,
        val_loader=test_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
        checkpoint_dir=RESULTS_DIR / f"fold_{fold_idx}",
        monitor_metric="auc",
    )

    test_loss, test_metrics = trainer.validate(verbose=True)
    test_results.append(
        {
            "fold": fold_idx,
            "test_loss": test_loss,
            **test_metrics,
        }
    )

#### Results

In [ ]:
test_results_df = pd.DataFrame(test_results)
test_results_df

In [ ]:
test_results_df.mean(numeric_only=True)